## Graph maker

### 1 - Import Libraries

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

In [2]:
# indicator(df, to_calculate, timeframe, weighter=False, correlator=False)
import Indicator
import Graph

### 2 - Import Data from Bitcoin Futures csv file and check the dataframe

In [3]:
df = pd.read_csv('bitcoin_futures_raw_data/futures_raw_data.csv')

In [4]:
df

,datetime,open_price,high_price,low_price,close_price,volume,buy_volume,transactions,buy_transactions,long_short_ratio,...,open_open_interest,high_open_interest,low_open_interest,close_open_interest,volume_delta,cumulative_volume_delta,transaction_delta,cumulative_transaction_delta,liquidation_delta,cumulative_liquidation_delta
0,2019-09-13,10414.96,10440.55,10153.51,10341.34,19260.071,9723.458,28011,14938.0,NaN,...,NaN,NaN,NaN,NaN,186.845,1.868450e+02,1865.0,1865.0,NaN,NaN
1,2019-09-14,10343.01,10419.97,10222.77,10332.25,20685.728,9537.192,28783,17393.0,NaN,...,NaN,NaN,NaN,NaN,-1611.344,-1.424499e+03,6003.0,7868.0,NaN,NaN
2,2019-09-15,10333.47,10359.20,10024.81,10302.22,20548.461,7433.957,25145,13365.0,NaN,...,NaN,NaN,NaN,NaN,-5680.547,-7.105046e+03,1585.0,9453.0,NaN,NaN
3,2019-09-16,10302.00,10353.81,10080.70,10249.27,19722.015,6842.919,28015,15181.0,NaN,...,NaN,NaN,NaN,NaN,-6036.177,-1.314122e+04,2347.0,11800.0,NaN,NaN
4,2019-09-17,10257.30,10270.63,10136.77,10186.52,22554.538,12686.993,28884,15076.0,NaN,...,NaN,NaN,NaN,NaN,2819.448,-1.032177e+04,1268.0,13068.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2219,2025-10-10,121579.40,122497.00,102648.40,112714.90,283074.793,132251.301,3255324,1602863.0,0.9040,...,96722.011,97084.623,72364.700,72439.474,-18572.191,-2.274482e+06,-49598.0,-2854672.0,749.450,215490.762
2220,2025-10-11,112715.00,113300.10,109501.00,110579.10,162986.706,73542.497,2041944,983670.0,1.1858,...,72434.743,76479.026,71421.781,74324.913,-15901.712,-2.290384e+06,-74604.0,-2929276.0,77.889,215568.651
2221,2025-10-12,110579.20,115730.30,109509.30,114894.40,214563.112,107056.890,2305725,1149944.0,1.4021,...,74331.358,76356.016,70503.009,75910.074,-449.332,-2.290833e+06,-5837.0,-2935113.0,24.561,215593.212
2222,2025-10-13,114894.50,115912.00,113600.50,115111.90,190121.618,92858.401,1858297,927903.0,1.5556,...,75909.323,77513.040,75673.319,76664.481,-4404.816,-2.295238e+06,-2491.0,-2937604.0,-2.659,215590.553


In [5]:
df.columns

Index(['datetime', 'open_price', 'high_price', 'low_price', 'close_price',
       'volume', 'buy_volume', 'transactions', 'buy_transactions',
       'long_short_ratio', 'long_liquidation', 'short_liquidation',
       'open_predicted_funding_rate', 'high_predicted_funding_rate',
       'low_predicted_funding_rate', 'close_predicted_funding_rate',
       'open_funding_rate', 'high_funding_rate', 'low_funding_rate',
       'close_funding_rate', 'open_open_interest', 'high_open_interest',
       'low_open_interest', 'close_open_interest', 'volume_delta',
       'cumulative_volume_delta', 'transaction_delta',
       'cumulative_transaction_delta', 'liquidation_delta',
       'cumulative_liquidation_delta'],
      dtype='object')

### 3 - Create the Graph Function

In [6]:
def nice_graph(
        df=df, 
        title='BTC/USD', 
        start='2019-09-13', 
        end='2025-10-14', 
        indicator_top='open_interest', 
        indicator_bot='volume', 
        timeframe_short=31, 
        timeframe_long=365, 
        top_type='candle', 
        bot_type='bar'
):
    
    # list of thing to calculate the indicators
    to_calculate_list = ['price', indicator_top, indicator_bot]

    # create indicators for long and short timeframes
    # first create 2 dataframes with datetime column
    df_indicators_short = df.copy()
    df_indicators_long = df.copy()

    # then add the indicator for each one
    for indicator in to_calculate_list:
        # fix the name of column if needed
        if  indicator  in ['price','predicted_funding_rate','funding_rate','open_interest']:
            indicator = f'close_{indicator}'
        # the column price and open interest must be weithed
        if 'price' in indicator or 'open_interest' in indicator:
            # Indicator.indicator format: indicator(df, to_calculate, timeframe, weighter=False)

            df_indicators_short =  df_indicators_short.merge(Indicator.indicator(df, indicator, timeframe_short, weighter='volume'), how='left', on='datetime')
            df_indicators_long =  df_indicators_long.merge(Indicator.indicator(df, indicator, timeframe_long, weighter='volume'), how='left', on='datetime')
        else:
            df_indicators_short =  df_indicators_short.merge(Indicator.indicator(df, indicator, timeframe_short), how='left', on='datetime')
            df_indicators_long =  df_indicators_long.merge(Indicator.indicator(df, indicator, timeframe_long), how='left', on='datetime')

        # gives the begining and end of the dataframe
    df_indicators_short = df_indicators_short[(df_indicators_short["datetime"] > start) & (df_indicators_short["datetime"] < end)]
    df_indicators_long = df_indicators_long[(df_indicators_long["datetime"] > start) & (df_indicators_long["datetime"] < end)] 

    ####################################################################################################################################################################
    # Figure base with 3 rows
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        row_heights=[0.2, 0.6, 0.2], 
        vertical_spacing=0.02,
        subplot_titles=(indicator_top.replace('_', ' ').title(), 'Price', indicator_bot.replace('_', ' ').title())
    )
    ####################################################################################################################################################################   
    # dict of type indicators and row
    all_indicators = {
        'top' : [top_type, indicator_top, 1], 
        'middle' : ['candle', 'price', 2],
        'bot': [bot_type, indicator_bot, 3]
    }

    # loop for each indicator and price
    for position, (type, indicator, row) in all_indicators.items():

        # create the dictionary with the conditions
        # if needs to fix the name of column
        if  indicator  in ['price','predicted_funding_rate','funding_rate','open_interest']: 
            # create the dictionary with the conditions  
            condition_dict =  Graph.cond(df_indicators_short, f'close_{indicator}', timeframe_short, 'z_score')
        else:
            # create the dictionary with the conditions
            condition_dict =  Graph.cond(df_indicators_short, indicator, timeframe_short, 'z_score')

        # (Call)
        # if its candle like 
        if type == 'candle':
            # color the candle by each condition
            for name, (condition, color) in condition_dict.items():
                Graph.candle(condition, indicator, name, color, row, fig)

        # if its line like 
        elif type == 'line':
            if  indicator  in ['price','predicted_funding_rate','funding_rate','open_interest']: 
                Graph.line(df_indicators_short, indicator, df[f'close_{indicator}'], 'gray', 2, row, fig)
            else: 
                Graph.line(df_indicators_short, indicator, df[indicator], 'gray', 2, row, fig)
        # if its bar like 
        elif type == 'bar':
            # color the bar by each condition
            for name, (condition, color) in condition_dict.items():
                Graph.bar(condition, indicator, name, color, row, fig)  

    ####################################################################################################################################################################
    # (Moving Averages and Bollinger Bands)
    # (calculate bolinger bands)

    # a dict where the data of bands will be stored with the color and the row 
    line_dict = {}

    # for each of indicator and price calculte the bands and ma then insert into the line dictionary
    for position, (type, indicator, row)  in all_indicators.items(): 

    # fixes the name of the column
        if  indicator  in ['price','predicted_funding_rate','funding_rate','open_interest']: 
            indicator = f'close_{indicator}'

        # (BAND)
        # price and open interest must be weithed
        if 'price' in indicator or 'interest' in indicator:
            df_bands = Indicator.band(df, indicator, timeframe_long, weighter='volume')
            
        # dont need to be weighted    
        else:
            df_bands = Indicator.band(df, indicator, timeframe_long)
        
        # gives the begining and end of the dataframe 
        df_bands = df_bands[(df_bands["datetime"] > start) & (df_bands["datetime"] < end)]

        # Take off low values
        ma_series = df_indicators_long[f'{indicator}_{timeframe_long}_ma'].reindex(df_bands.index)

        for col in df_bands.columns:
            if col != 'datetime':
                df_bands[col] = df_bands[col].where(df_bands[col] > (ma_series / 3), np.nan)
                
        # for each band
        for band in df_bands.columns:            
            # datetime should not be used to create a line
            if band != 'datetime':
                # separetes upper from lower
                if 'upper' in band:
                    if '_2_' in band:
                        line_dict[f'Upper_{indicator.replace('close_', '')}_band_2'] = [df_bands[band], "salmon", 0.4, row]

                # create lower lines for types diffent to bar
                elif type != 'bar' and 'lower' in band:
                    if '_2_' in band:
                        line_dict[f'Lower_{indicator.replace('close_', '')}_band_2'] = [df_bands[band], "skyblue", 0.4, row]

        # (MA)
        line_dict[f'MA_{indicator.replace('close_', '')}_{timeframe_long}'] = [df_indicators_long[f'{indicator}_{timeframe_long}_ma'], 'indigo', 1, row]
        line_dict[f'MA_{indicator.replace('close_', '')}_{timeframe_short}'] = [df_indicators_short[f'{indicator}_{timeframe_short}_ma'], "goldenrod", 1, row]

    # (Call)    
    for name, (line, color, width, row) in line_dict.items():
        Graph.line(df_indicators_long, name, line, color, width, row, fig)

    ####################################################################################################################################################################
    # (Final Steps)
    #  adjust the axes
    fig.update_yaxes(showgrid=False,autorange=True, fixedrange=False,row=1, col=1) # remove grid for row 1
    fig.update_yaxes(showgrid=False,autorange=True, fixedrange=False, row=2, col=1)  # remmove grid for row 2
    fig.update_yaxes(showgrid=False, row=3, col=1) # remove grid for row 3

    # create 
    fig.update_layout(    
        plot_bgcolor="#212121", #background color
        paper_bgcolor="#212121", #outside color
        font=dict(color="white"), # font color
        xaxis=dict(showgrid=False), # remove the grid for every plot
        xaxis2=dict(showgrid=False),
        xaxis3=dict(showgrid=False),
        yaxis=dict(showgrid=False),
        title=title, # title
        xaxis_rangeslider_visible=False, # remove range slider
        xaxis2_rangeslider_visible=False,
        xaxis3_rangeslider_visible=False,
        dragmode='zoom', # permites zooming
        annotations=[
            dict(text=indicator_top.replace('_', ' ').title(), x=0.01, y=1.05, xref='paper', yref='paper',
                xanchor='left', showarrow=False, font=dict(size=12, color='white')),
            dict(text='Price', x=0.01, y=0.70, xref='paper', yref='paper',
                xanchor='left', showarrow=False, font=dict(size=12, color='white')),
            dict(text=indicator_bot.replace('_', ' ').title(), x=0.01, y=0.35, xref='paper', yref='paper',
                xanchor='left', showarrow=False, font=dict(size=12, color='white')),
        ]
    )
    
    # finaly return the figure
    return fig

In [7]:
# (INDICATORS AND TYPE SUPPORTED)
###
#    'volume', (bar)      
#    'volume_delta', (bar)                   
#    'cumulative_volume_delta', (line)       
#    'transactions', (bar)                    
#    'transaction_delta', (bar)                
#    'cumulative_transaction_delta', (line)  
#    'long_short_ratio', (line)
#    'short_liquidation', (line)
#    'long_liquidation', (bar)
#    'liquidation_delta', (bar)
#    'cumulative_liquidation_delta', (line)
#    'predicted_funding_rate', (candle, line)
#    'funding_rate', (candle, line)
#    'open_interest' (candle, line)
###
# call the function 
fig = nice_graph(
        df=df, 
        title='BTC/USD color by Z-Score', 
        start='2025-01-01', 
        end='2025-10-14', 
        indicator_top='open_interest', 
        indicator_bot='volume_delta', 
        timeframe_short=31, # short window time series
        timeframe_long=365, # long window time series
        top_type='candle', 
        bot_type='bar'
)

fig.show()

In [8]:
fig.write_html(f"figure_test.html")